# Physical variables → components → scores → bins

This notebook follows the highest-level public workflow without hiding it inside an example runner. A two-variable event model has three additive intensity components. ScoreQuant evaluates the components, constructs coefficient scores, and learns eight hard bins.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples._env import example_scale
from examples.linear_workflow import background_1, background_2, signal

rng = np.random.default_rng(17)
n_mc = example_scale(2_000, 400)
X_mc = np.column_stack([rng.uniform(0, 1, n_mc), rng.uniform(-1, 1, n_mc)])
mc_weights = 0.5 + rng.random(len(X_mc))
X_mc.shape

## Declare the scientific model

For $\lambda(x;c)=\sum_k c_k\phi_k(x)$, the coefficient score is $s_k(x)=\phi_k(x)/\lambda(x;c_0)$. `LinearComponents` stores named component callables, their reference coefficients, and variable names, so the same frozen model can convert new physical variables to scores again at prediction time.

In [ ]:
model = sq.LinearComponents(
    components={"signal": signal, "background_1": background_1, "background_2": background_2},
    coefficients={"signal": 1.0, "background_1": 0.4, "background_2": 0.2},
    variables=["energy", "cos_theta"],
)
component_values = np.asarray(model.evaluate_components(X_mc))
component_values.shape, component_values[:2]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)
for index, (axis, name) in enumerate(zip(axes, model.component_names, strict=True)):
    points = axis.scatter(X_mc[:, 0], X_mc[:, 1], c=component_values[:, index], s=6, cmap="viridis")
    axis.set(title=name, xlabel="energy", ylabel="cos(theta)")
    fig.colorbar(points, ax=axis)

## Fit from physical variables

`fit_quantizer` always consumes an explicit source. `ObservationSample` pairs physical variables with `LinearComponentScore`; prediction keeps observation-to-score conversion visible.

In [ ]:
provider = sq.LinearComponentScore(model)
result = sq.fit_quantizer(
    sq.ObservationSample(X_mc, mc_weights),
    score=provider,
    n_bins=8,
    criterion=sq.NormalizedTrace(),
    config=sq.KMeansConfig(seed=42, n_init=4),
)
print(result.report())

In [ ]:
n_data = example_scale(500, 100)
X_data = np.column_stack([rng.uniform(0, 1, n_data), rng.uniform(-1, 1, n_data)])
data_labels = np.asarray(result.predict_scores(provider.score(X_data)))
counts = np.bincount(data_labels, minlength=result.n_bins)
counts

## Output contract

There is no `result.predict` that accepts physical variables directly. Prediction is always the explicit `predict_scores`: the cell above calls `provider.score(X_data)` to convert new physical variables into scores with the same frozen model used for fitting, then passes those scores to `result.predict_scores`. Observation-to-score conversion is never hidden inside prediction. The eight counts are ready for an application likelihood; fitting the three coefficients from those counts is deliberately outside ScoreQuant.

### Why partitions do not predict

`fit_quantizer` returns a `QuantizerResult` like the one above: a reusable score-space rule with `predict_scores`. The companion task, `optimize_partition`, instead returns a `PartitionResult` that assigns only the fixed sample it was given — it deliberately has no `predict_scores` method, because a finite exchange-optimized assignment is not automatically a rule that generalizes to unseen events. The one exception is an exchange-stable, nonsingular D-optimal result, which can call `compile_quantizer()` to produce a theorem-backed Mahalanobis rule that does predict.